# Validação do ambiente U-Mamba (estágio 11)

Este notebook prepara um runtime atual do Google Colab para executar a arquitetura oficial `UMambaEnc_2d` e realiza um smoke test 256×256.

**Antes de executar:** em `Ambiente de execução → Alterar tipo de ambiente de execução`, selecione uma **GPU NVIDIA** (T4 ou outra disponível). O Colab não permite ativar a GPU por código.

## 1. Bootstrap do TCC

Carrega a versão atual do código do repositório.

In [ ]:
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## 2. Confirmar GPU

Interrompe imediatamente se o notebook ainda estiver em CPU.

In [ ]:
import platform
import torch

print(f"Sistema: {platform.platform()}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA do PyTorch: {torch.version.cuda}")
print(f"CUDA disponível: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU não está ativa. No Colab: Ambiente de execução > Alterar tipo de ambiente de execução "
        "> selecione GPU. Depois reinicie e execute desde a primeira célula."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


## 3. Preparar Mamba + arquitetura oficial

Instala as dependências necessárias somente se estiverem ausentes, valida um bloco `Mamba` na GPU e prepara uma cópia de runtime do arquivo oficial `UMambaEnc_2d.py` em um commit fixado. A adaptação remove apenas a integração de planejamento do nnU-Net que não é necessária para instanciar a rede diretamente.

A primeira execução pode demorar alguns minutos porque extensões CUDA podem ser compiladas.

In [ ]:
import json

from src.models.umamba_runtime import ensure_umamba_runtime

runtime_info = ensure_umamba_runtime()
print(json.dumps(runtime_info, indent=2, ensure_ascii=False))


## 4. Smoke test U-Mamba RGB

Constrói `UMambaEnc_2d` com entrada RGB e executa uma inferência sintética. O teste só passa se a saída mantiver 256×256 pixels e um canal de segmentação.

In [ ]:
from src.config import get_config
from src.models.umamba import build_official_umamba_enc_2d

config = get_config()
features = tuple(int(v) for v in config["model"]["umamba_features"])
device = torch.device("cuda")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = build_official_umamba_enc_2d(
    input_channels=3,
    num_classes=1,
    input_size=(256, 256),
    features_per_stage=features,
).to(device)

x = torch.randn(1, 3, 256, 256, device=device)
with torch.inference_mode():
    y = model(x)

peak_vram_mb = torch.cuda.max_memory_allocated() / 1024**2
parameters = sum(p.numel() for p in model.parameters())

print(f"Entrada: {tuple(x.shape)}")
print(f"Saída: {tuple(y.shape)}")
print(f"Parâmetros: {parameters:,}")
print(f"Pico de VRAM no smoke test: {peak_vram_mb:.1f} MB")

if tuple(y.shape) != (1, 1, 256, 256):
    raise RuntimeError(f"Shape de saída inesperado: {tuple(y.shape)}")

print("SMOKE TEST U-MAMBA: OK")


## 5. Salvar relatório do ambiente

Registra no Drive a configuração que realmente passou no teste.

In [ ]:
from src import io

storage_paths = io.resolve_storage_paths()
runs_dir = storage_paths["artifacts_runs"]
runs_dir.mkdir(parents=True, exist_ok=True)

report = {
    **runtime_info,
    "input_shape": list(x.shape),
    "output_shape": list(y.shape),
    "parameters": parameters,
    "smoke_test_peak_vram_mb": peak_vram_mb,
    "status": "ok",
}
report_path = runs_dir / "umamba_environment.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Relatório salvo em: {report_path}")


## Critério para seguir ao notebook 12

Só prossiga se a célula do smoke test terminar com **`SMOKE TEST U-MAMBA: OK`** e o relatório `umamba_environment.json` for salvo no Drive.